In [1]:
import sys
from pathlib import Path
# Production layout: add project root and src for imports (run from repo root or notebooks/ingestion/)
_root = Path(".").resolve()
if _root.name == "ingestion":
    _root = _root.parent.parent
elif (_root / "src").is_dir():
    pass
else:
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "src"))

from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries
from storage.cloud.CloudStorage import CloudStorageProvider

import pandas as pd
from datetime import datetime

In [2]:
table_name = PostgresSQL_table_queries.FINANCIAL_NEWS_TABLE_NAME
pg_conn = PgConn(table_name)
df = pg_conn.get_financial_news()
if df is None:
    raise RuntimeError(
        "get_financial_news() failed — check the error printed above "
        "(connection, table name, or schema)."
    )

Connection to the database successful!
Table name set to: financial_news_241118


In [3]:
print(f"Total news articles queried: {df.shape[0]}")

Total news articles queried: 10080


In [4]:
df.head()

,id,source,headline,href,summary,content,author,minsread,datetime
0,45876047236640910017045281208231353781,Motley Fool,"""Tokenized"" Stocks Are Breaking Down Barriers....",https://finance.yahoo.com/news/tokenized-stock...,,In This Article:\nHOOD\n+0.23%\nAAPL\n-0.12%\n...,"Alex Carchidi, The Motley Fool",6 min read,2025-07-23T17:19:00.000Z
1,220799662638683730910896676542146227596,CoinDesk,"""We're a Tweet Away from a 15% Dump:"" CEO on C...",https://finance.yahoo.com/video/were-tweet-awa...,,"In today's Markets Outlook, CoinDesk's Jennife...",CoinDesk,,2025-10-31T17:34:16.000Z
2,47514014236007877639078711690178672970,Motley Fool,$1 Billion in New Capital Could Soon Flow to X...,https://finance.yahoo.com/news/1-billion-capit...,,In this article:\nXRP-USD\n-0.06%\nKey Points\...,"Alex Carchidi, The Motley Fool",5 min read,2025-11-02T11:35:00.000Z
3,152599983586513563130066357372522406611,Coinspeaker,"$1.2B Left BTC ETFs This Week, but Not All is ...",https://finance.yahoo.com/news/1-2b-left-btc-e...,,In this article:\nBTC-USD\n+0.22%\nSOSO-USD\n+...,Parth Dubey,2 min read,2025-10-18T10:15:49.000Z
4,214287918542718613358470608080600126044,TheStreet,$1.5M Bitcoin prediction firm Ark Invest calls...,https://finance.yahoo.com/news/1-5m-bitcoin-pr...,According to ARK Invest's latest Bitcoin month...,"The Cathie Wood-owned firm believes Bitcoin, w...",Anushka Basu,2 min read,2025-03-12T16:42:53.000Z


In [5]:
def delete_records_for_current_date(df, pg_conn):
    if df is None:
        print("DataFrame is empty. No records to delete.")
        return
        
    # Filter DataFrame for records with today's date
    today = datetime.today().date()
    dt = pd.to_datetime(df['datetime'], errors='coerce')
    today_records = df[dt.dt.date == today]

    # Extract date strings (DB/delete API expects original string form)
    date_strings = today_records['datetime'].astype(str).tolist()

    # Call the delete_records_by_date method
    pg_conn.delete_records_by_date(date_strings)

# Then call the delete_records_for_current_date method
#delete_records_for_current_date(df, pg_conn)
#list_ids = ["11111"]
#pg_conn.delete_records_by_ids(list_ids)

In [6]:
class DataETL():
    
    def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
    class Export():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def set_dataframe(self, dataframe):
            self.df = dataframe
        
        def export_text_to_s3(self, bucket_name, prefix_path, file_format):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()

            # Create a new bucket
            aws_storage.create_bucket(bucket_name)

            # Upload DataFrame with datetime subfolder structure
            aws_storage.upload_dataframe_with_datetime_subfolders(self.df, bucket_name, prefix_path, file_format)
        
        def export_text_to_s3_full_file(self, bucket_name, prefix_path, filename):
            aws_storage = self.cloudProvider.AWS()
            aws_storage.upload_dataframe_to_csv(self.df, bucket_name, filename, prefix_path)
            
    class Ingestion():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def get_full_data_csv_file(self, bucket_name, prefix_path):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_csv_from_specific_folder(bucket_name, prefix_path)
        
        def get_data_csv_file_by_datetime(self, bucket_name, prefix_path, year, month, day, hour, minute):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_dataframe_from_specific_datetime(bucket_name, prefix_path, year=None, month=None, day=None, hour=None, minute=None)
    
    class Process():
        
        def __init__(self, dataframe):
            self.df = dataframe
        
        def getData():
            self.df = pg_conn.get_financial_news()
        
        def filter_by_current_date(self):
            today = datetime.today().date()
            dt = pd.to_datetime(self.df['datetime'], errors='coerce')
            return self.df[dt.dt.date == today]
            
    class Transform():
        def extractStopWords():
            pass

In [7]:
etl = DataETL(df)
etl_process = etl.Process(etl.df)
filtered_df = etl_process.filter_by_current_date()
filtered_df.head()

,id,source,headline,href,summary,content,author,minsread,datetime
71,112271484810819069292170596627025453439,decrypt,'Private Bitcoin' to Launch on Starknet With Z...,https://finance.yahoo.com/news/private-bitcoin...,,In this article:\nSTRK22691-USD\n-4.10%\nBTC-U...,André Beganski,3 min read,2026-02-26T23:01:12.000Z
93,79567273327088534030282011458120978368,Motley Fool,"1 Cryptocurrency to Consider Buying With $2,00...",https://finance.yahoo.com/news/1-cryptocurrenc...,,In this article:\nDOGE-USD\n-4.23%\nNVDA\n-5.4...,"Alex Carchidi, The Motley Fool",4 min read,2026-02-26T05:50:00.000Z
748,29911906174093391575825846216891467593,TheStreet,Another crypto executive accused of insider tr...,https://finance.yahoo.com/news/another-crypto-...,,In this article:\nBTC-USD\n-1.30%\nPopular onc...,Anand Sinha,3 min read,2026-02-26T17:23:15.000Z
941,316878550224202608217373602922154345297,24/7 Wall St.,BITI Shorts Bitcoin And Popped 25%. Now the Tr...,https://finance.yahoo.com/news/biti-shorts-bit...,,In this article:\nBTC-USD\n-1.41%\nQuick Read\...,Michael Williams,3 min read,2026-02-26T18:31:22.000Z
1204,312205210779296090874806522702446593963,Reuters,Binance cannot arbitrate customer claims over ...,https://finance.yahoo.com/news/binance-cannot-...,,"NEW YORK, Feb 26 (Reuters) - A federal judge o...",Reuters,1 min read,2026-02-26T23:39:45.000Z


In [8]:
print(f"Total filtered news articles queried: {filtered_df.shape[0]}")

Total filtered news articles queried: 49


In [9]:
export_rows_to_s3 = True
etl_export = etl.Export(filtered_df)
bucket_name = "test-financial-news-bucket"
prefix_path = "news/crypto"
file_format = "csv"
if export_rows_to_s3 == True and not filtered_df.empty:
    etl_export.export_text_to_s3(bucket_name, prefix_path, file_format)

Bucket 'test-financial-news-bucket' created successfully.
Data for row 71 with id '112271484810819069292170596627025453439' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=02/day=26/hour=23/minute=01/second=12/format=csv/112271484810819069292170596627025453439.csv'
Data for row 93 with id '79567273327088534030282011458120978368' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=02/day=26/hour=05/minute=50/second=00/format=csv/79567273327088534030282011458120978368.csv'
Data for row 748 with id '29911906174093391575825846216891467593' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=02/day=26/hour=17/minute=23/second=15/format=csv/29911906174093391575825846216891467593.csv'
Data for row 941 with id '316878550224202608217373602922154345297' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=02/day=26/hour=18/minute=3

Data for row 6779 with id '86692656526731852253995490097593378898' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=02/day=26/hour=15/minute=03/second=12/format=csv/86692656526731852253995490097593378898.csv'
Data for row 7555 with id '98385637262610656159433038049773707344' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=02/day=26/hour=15/minute=45/second=39/format=csv/98385637262610656159433038049773707344.csv'
Data for row 8121 with id '250152591070998114758262892532058528046' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=02/day=26/hour=09/minute=36/second=14/format=csv/250152591070998114758262892532058528046.csv'
Data for row 8298 with id '179095415923007089299555914531049064910' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=02/day=26/hour=22/minute=00/second=03/format=csv/17909541592300708929955591453

In [10]:
post_full_csv = False
if (post_full_csv == True) and not filtered_df.empty:
    now = datetime.now()
    filename = f"{now.year}-{now.month:02}-{now.day:02}_full_record"
    etl_export.set_dataframe(etl.df)
    etl_export.export_text_to_s3_full_file(bucket_name, prefix_path, filename)

In [11]:
ingest_data = False
get_full_file = True
get_by_datetime = False
df_from_file = None

if ingest_data == True:
    etl_ingestion = etl.Ingestion(etl.df)
    bucket_name = "test-financial-news-bucket"
    prefix_path = "news/crypto/"
    year = '2024'
    month = '08'
    day = '01'
    hour = ''
    minute = ''
    if get_full_file == True:
        filename = f"{year}-{month}-{day}_full_record.csv"
        full_path = f"{prefix_path}{filename}"
        df_from_file = etl_ingestion.get_full_data_csv_file(bucket_name, full_path)
    elif get_by_datetime == True:
        df_from_file = etl_ingestion.get_data_csv_file_by_datetime(bucket_name, prefix_path, year, month, day, hour, minute)

In [12]:
if df_from_file is not None:
    print(df_from_file.count())

In [13]:
# Assuming df is your DataFrame
targetId = ''
if len(targetId) > 0:
    filtered_content = filtered_df.loc[filtered_df['id'] == targetId, 'content']

    # If you want to display the full content, convert it to a list or display all rows
    full_content = filtered_content.tolist()

    print(full_content)